# Practical Reranking Implementation: Cross-Encoders and FlashRank (Lecture 45)

This notebook implements the second stage of the Three-Stage RAG architecture: **Reranking**.
While fast retrievers (bi-encoders / sparse search) retrieve candidate documents rapidly ($O(1)$ to $O(\log N)$), they evaluate query and document representations independently.
A **Reranker** (Cross-Encoder / FlashRank) processes query and candidate text jointly via full self-attention, generating precise relevance scores and correcting retrieval ordering.

### Architectural Workflow
1. **Stage 1 (Broad Retrieval)**: Retrieve top-K candidates using an in-memory dense vector store (FAISS).
2. **Stage 2 (Joint Reranking)**: Rescore candidates using `sentence_transformers.CrossEncoder` and `flashrank.Ranker`.
3. **Stage 3 (Augmented Generation)**: Pass top-N reranked context into an LCEL chain with `ChatGroq`.

## 1. Setup & Environment Initialization
Load dependencies and initialize local embeddings and reranking models.

In [1]:
import os
import time
from typing import List, Dict, Any, Tuple
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

from sentence_transformers import CrossEncoder
from flashrank import Ranker, RerankRequest

load_dotenv()

# Initialize local dense embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Initialize Cross-Encoder and FlashRank models
cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
flashrank_ranker = Ranker(model_name="ms-marco-TinyBERT-L-2-v2")

print("All embedding and reranking models initialized successfully.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_2664\3023944482.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

All embedding and reranking models initialized successfully.


## 2. Ingesting Multi-Domain Corpus with Lexical Distractors
Construct a corpus containing relevant documents alongside deceptive distractors that share lexical overlap.

In [2]:
corpus_data: List[Document] = [
    Document(
        page_content="Python performance optimization involves using vectorized operations in NumPy and JIT compilation via Numba.",
        metadata={"id": "doc_python_opt", "topic": "programming"}
    ),
    Document(
        page_content="Python was created by Guido van Rossum and released in 1991 as a general-purpose programming language emphasizing code readability.",
        metadata={"id": "doc_python_history", "topic": "history"}
    ),
    Document(
        page_content="To accelerate Python execution in production RAG systems, developers employ asynchronous I/O and multithreaded vector indexing.",
        metadata={"id": "doc_rag_python_perf", "topic": "rag_infra"}
    ),
    Document(
        page_content="High-speed bullet trains in Japan achieve speeds over 300 km/h using magnetic levitation and aerodynamic design.",
        metadata={"id": "doc_trains", "topic": "transport"}
    ),
    Document(
        page_content="Reranking is an effective technique to rescore retrieved documents in a RAG pipeline using deep transformer cross-attention.",
        metadata={"id": "doc_reranking_core", "topic": "rag_theory"}
    ),
    Document(
        page_content="Cross-encoders take both query and passage simultaneously as a single concatenated input sequence to compute joint interaction scores.",
        metadata={"id": "doc_cross_encoder_mech", "topic": "reranker_mechanics"}
    ),
    Document(
        page_content="FlashRank provides ultra-lightweight ONNX-accelerated reranking requiring minimal memory footprint for serverless microservices.",
        metadata={"id": "doc_flashrank_mech", "topic": "flashrank"}
    ),
]

vector_store = FAISS.from_documents(corpus_data, embeddings)
print(f"FAISS index built with {vector_store.index.ntotal} documents.")

INFO:faiss.loader:Loading faiss with AVX2 support.


INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")


INFO:faiss.loader:Loading faiss.


INFO:faiss.loader:Successfully loaded faiss.


FAISS index built with 7 documents.


## 3. Comparing Stage 1 Dense Retrieval vs. Stage 2 Cross-Encoder Reranking
Query the vector store for candidates and demonstrate rank inversion via Cross-Encoder.

In [3]:
query = "How does cross-encoder compute joint interaction between query and passage?"

# Stage 1: Dense Retrieval (Broad candidates k=5)
dense_candidates: List[Document] = vector_store.similarity_search(query, k=5)

print("=== Stage 1: Initial Dense Retrieval Ranking ===")
for rank, doc in enumerate(dense_candidates, start=1):
    print(f"Rank {rank} [{doc.metadata['id']}]: {doc.page_content[:80]}...")

# Stage 2: Cross-Encoder Rescoring
pairs: List[Tuple[str, str]] = [(query, doc.page_content) for doc in dense_candidates]
start_ce = time.perf_counter()
ce_scores = cross_encoder_model.predict(pairs)
ce_latency = (time.perf_counter() - start_ce) * 1000

scored_candidates = list(zip(dense_candidates, ce_scores))
reranked_candidates = sorted(scored_candidates, key=lambda x: x[1], reverse=True)

print(f"\n=== Stage 2: Cross-Encoder Reranked Results ({ce_latency:.2f} ms) ===")
for rank, (doc, score) in enumerate(reranked_candidates, start=1):
    print(f"Rank {rank} [Score: {score:+.4f}] [{doc.metadata['id']}]: {doc.page_content}")

top_reranked_doc, top_score = reranked_candidates[0]
assert top_reranked_doc.metadata["id"] == "doc_cross_encoder_mech", "Cross-Encoder must rank the exact explanatory document first"
assert top_score > 0.0, "Relevance score of top document must be positive"
print("\nCross-Encoder rescoring assertion passed.")

=== Stage 1: Initial Dense Retrieval Ranking ===
Rank 1 [doc_cross_encoder_mech]: Cross-encoders take both query and passage simultaneously as a single concatenat...
Rank 2 [doc_reranking_core]: Reranking is an effective technique to rescore retrieved documents in a RAG pipe...
Rank 3 [doc_rag_python_perf]: To accelerate Python execution in production RAG systems, developers employ asyn...
Rank 4 [doc_python_opt]: Python performance optimization involves using vectorized operations in NumPy an...
Rank 5 [doc_flashrank_mech]: FlashRank provides ultra-lightweight ONNX-accelerated reranking requiring minima...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== Stage 2: Cross-Encoder Reranked Results (31.82 ms) ===
Rank 1 [Score: +8.8602] [doc_cross_encoder_mech]: Cross-encoders take both query and passage simultaneously as a single concatenated input sequence to compute joint interaction scores.
Rank 2 [Score: -10.8815] [doc_reranking_core]: Reranking is an effective technique to rescore retrieved documents in a RAG pipeline using deep transformer cross-attention.
Rank 3 [Score: -11.2793] [doc_python_opt]: Python performance optimization involves using vectorized operations in NumPy and JIT compilation via Numba.
Rank 4 [Score: -11.3016] [doc_rag_python_perf]: To accelerate Python execution in production RAG systems, developers employ asynchronous I/O and multithreaded vector indexing.
Rank 5 [Score: -11.4054] [doc_flashrank_mech]: FlashRank provides ultra-lightweight ONNX-accelerated reranking requiring minimal memory footprint for serverless microservices.

Cross-Encoder rescoring assertion passed.


## 4. Ultra-Fast Lightweight Reranking with FlashRank
Evaluate FlashRank ONNX inference latency and ranking consistency.

In [4]:
flashrank_passages = [
    {"id": doc.metadata["id"], "text": doc.page_content}
    for doc in dense_candidates
]

req = RerankRequest(query=query, passages=flashrank_passages)
start_fr = time.perf_counter()
flashrank_results = flashrank_ranker.rerank(req)
fr_latency = (time.perf_counter() - start_fr) * 1000

print(f"FlashRank Execution Latency: {fr_latency:.2f} ms\n")
for rank, item in enumerate(flashrank_results, start=1):
    print(f"Rank {rank} [Score: {item['score']:.4f}] [{item['id']}]: {item['text'][:80]}...")

assert flashrank_results[0]["id"] == "doc_cross_encoder_mech", "FlashRank top result must match expected target document"
assert fr_latency < 500.0, "FlashRank should operate within low-latency bounds"
print("FlashRank reranking assertion passed.")

FlashRank Execution Latency: 2.38 ms

Rank 1 [Score: 0.9999] [doc_cross_encoder_mech]: Cross-encoders take both query and passage simultaneously as a single concatenat...
Rank 2 [Score: 0.0000] [doc_reranking_core]: Reranking is an effective technique to rescore retrieved documents in a RAG pipe...
Rank 3 [Score: 0.0000] [doc_python_opt]: Python performance optimization involves using vectorized operations in NumPy an...
Rank 4 [Score: 0.0000] [doc_flashrank_mech]: FlashRank provides ultra-lightweight ONNX-accelerated reranking requiring minima...
Rank 5 [Score: 0.0000] [doc_rag_python_perf]: To accelerate Python execution in production RAG systems, developers employ asyn...
FlashRank reranking assertion passed.


## 5. End-to-End Three-Stage RAG Pipeline Integration (LCEL + ChatGroq)
Assemble a complete RAG chain where the Reranker filters top candidates prior to LLM generation.

In [5]:
def rerank_filter(docs: List[Document], query: str, top_n: int = 2) -> List[Document]:
    """Filter retrieved documents using Cross-Encoder scores."""
    pairs = [(query, d.page_content) for d in docs]
    scores = cross_encoder_model.predict(pairs)
    ranked = [doc for doc, _ in sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)]
    return ranked[:top_n]

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

def three_stage_retriever(query_text: str) -> str:
    raw_candidates = retriever.invoke(query_text)
    refined_docs = rerank_filter(raw_candidates, query_text, top_n=2)
    return "\n\n".join(f"[{d.metadata['id']}]: {d.page_content}" for d in refined_docs)

prompt = ChatPromptTemplate.from_template("""You are an AI research specialist. Answer based strictly on the context provided.
Context:
{context}

Question:
{question}

Answer:""")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

rag_chain = (
    {"context": RunnablePassthrough(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

target_question = "Explain how cross-encoders process query and passage pairs together."
refined_context = three_stage_retriever(target_question)
final_answer = rag_chain.invoke({"context": refined_context, "question": target_question})

print(f"Question: {target_question}\n")
print(f"Refined Context Injected to LLM:\n{refined_context}\n")
print(f"LLM Generated Answer:\n{final_answer.strip()}")

assert "simultaneous" in final_answer.lower() or "concatenat" in final_answer.lower() or "single" in final_answer.lower() or "joint" in final_answer.lower(), "Answer must reflect joint cross-attention mechanism"
print("\nEnd-to-end Three-Stage RAG pipeline assertion passed.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Question: Explain how cross-encoders process query and passage pairs together.

Refined Context Injected to LLM:
[doc_cross_encoder_mech]: Cross-encoders take both query and passage simultaneously as a single concatenated input sequence to compute joint interaction scores.

[doc_reranking_core]: Reranking is an effective technique to rescore retrieved documents in a RAG pipeline using deep transformer cross-attention.

LLM Generated Answer:
Cross‑encoders handle a query–passage pair by feeding **both** texts into the model at the same time.  
1. **Concatenation** – The query and the passage are joined into one single input sequence (often with a separator token).  
2. **Joint encoding** – This concatenated sequence is passed through a deep transformer that uses cross‑attention across the entire sequence.  
3. **Interaction score** – The transformer’s final hidden states are pooled (e.g., CLS token) and transformed into a single relevance score that reflects the joint interaction betwee